# Coastal Water Quality Measurements at an Atlantic Sea Bass Farm in Sines, Portugal (2018–2019) Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.qkeb-wqrx) coastal water quality dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. This approach utilizes the [Croissant schema](https://mlcommons.org/croissant/) to load dataset metadata, records, and to perform exploratory data analysis, referencing all record sets and fields by their `@id`.

### Dataset Source
The dataset schema is provided via a Croissant JSON-LD URL.

In [ ]:
# Install mlcroissant if not present
!pip install mlcroissant --quiet

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`. The dataset will be loaded by providing the URL to the Croissant schema.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qkeb-wqrx/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print main dataset metadata fields
metadata = dataset.metadata
print("Name:", metadata.name)
print("Description:", metadata.description)
print("@id:", metadata.id)
print("Keywords:", getattr(metadata, 'keywords', []))

## 2. Data Overview
Let's explore the available record sets and their fields. Each record set, field, and column is referenced by its `@id`, according to the Croissant schema.

Below, all `@id`s of record sets are listed. For each record set, fields are described. This can help you decide what data to analyze!

In [ ]:
print("Available record sets (referenced by their @id):\n")
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # mlcroissant 0.2+ uses 'recordSets'
    record_sets = getattr(metadata, 'recordSets', [])
record_set_ids = []
for rset in record_sets:
    print(f"- {rset.id}: {getattr(rset, 'name', '')}")
    record_set_ids.append(rset.id)
    # List all fields in each record set
    if hasattr(rset, 'fields'):
        print("  Fields (by @id):")
        for field in rset.fields:
            print(f"    - {field.id} ({field.name}), type: {getattr(field, 'data_type', '?')}")
    else:
        print("  No fields defined.")
    print("")
if not record_set_ids:
    print("No record sets found in this Croissant schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we attempt to load all available record sets. Each DataFrame's columns are field `@id`s for clarity and consistency.

In [ ]:
dataframes = {}
print("Loading records from all record sets...\n")
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist() if not df.empty else 'No records'}\n")
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")

# To proceed, pick a record set with data.
if dataframes:
    # Pick first non-empty dataframe
    selected_record_set_id = None
    for k, df in dataframes.items():
        if not df.empty:
            selected_record_set_id = k
            break
    if selected_record_set_id is not None:
        print(f"Selected record set for further analysis: {selected_record_set_id}")
        print("Column @ids:")
        print(dataframes[selected_record_set_id].columns.tolist())
        dataframes[selected_record_set_id].head()
    else:
        print("No non-empty record sets found.")
else:
    print("No record sets could be loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA operations by referencing fields by their `@id`. We will attempt to select a numeric field from the DataFrame for demonstration purposes.

Note: Since field names may be non-human readable `@id`s, refer to the schema cell above or the DataFrame columns for the correct `@id`.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Example: Use the selected non-empty record set and pick a numeric field
if dataframes:
    df = dataframes[selected_record_set_id].copy()
    numeric_field = None
    for col in df.columns:
        # Try to auto-detect a numeric field by checking dtype conversion
        try:
            df[col] = pd.to_numeric(df[col])
            if np.issubdtype(df[col].dtype, np.number) and not df[col].isnull().all():
                numeric_field = col
                break
        except Exception:
            continue
    if numeric_field is None:
        print("No numeric fields found in the selected record set.")
    else:
        print(f"Using numeric field: {numeric_field} (referenced by @id)")

        # Filter values above an arbitrary threshold (example: the mean)
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a field (e.g., first column that's not numeric_field)
        group_field = None
        for col in df.columns:
            if col != numeric_field:
                group_field = col
                break
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No data to analyze.")

## 5. Visualization
Plot data distributions or the relationship between numeric and categorical fields, referencing them by their record set and field `@id`s.

Below, a histogram and boxplot of the numeric field (by `@id`) are shown if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None and not df[numeric_field].isnull().all():
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    
    plt.subplot(1, 2, 2)
    sns.boxplot(y=df[numeric_field])
    plt.title(f"Boxplot of {numeric_field}")

    plt.tight_layout()
    plt.show()

    # If group_field is categorical, plot mean numeric_field by group
    if group_field:
        group_means = df.groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field, y=numeric_field, data=group_means)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("Not enough numeric data for visualization.")

## 6. Conclusion
In this notebook, you loaded and explored the FAIR^2 Coastal Water Quality dataset using `mlcroissant`. All references to data entities (record sets, fields) were made using their `@id` per the Croissant schema, supporting traceability and automated workflows.

- You loaded metadata and described the dataset and its record sets/fields from the Croissant schema URL.
- You extracted tabular data using the `@id` of a record set and fields, and performed basic EDA (filtering, normalization, grouping).
- You visualized the distribution and group means of a numeric field, all referencing field and record set `@id`.

You can now adapt and extend this notebook to process additional record sets, link fields by their Croissant `@id`, or build data analysis pipelines using the FAIR Schema!